Loading the Data and Exploring

In [2]:
!pip install requests
!pip install aria2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 79.7 MB/s eta 0:00:00


In [3]:
import numpy as np, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import random
from collections import defaultdict
import pickle
import numpy as np
import itertools
import subprocess
import requests

In [4]:
# Get download URL first
record_id = "17686067"
api_url = f"https://zenodo.org/api/records/{record_id}"
record = requests.get(api_url).json()

# Download with aria2c (16 parallel connections)
for file_info in record['files']:
    if file_info['key'].endswith('.pkl'):
        url = file_info['links']['self']
        filename = file_info['key']

        print(f"Downloading {filename} with aria2c...")
        subprocess.run([
            'aria2c',
            '-x16',  # 16 connections
            '-s16',  # 16 splits
            '--file-allocation=none',
            f'--out={filename}',
            url
        ])
        break

In [18]:
with open("/content/processed_mosei.pkl", "rb") as f:
  data = pickle.load(f)

In [19]:
print(f"There are {len(data.keys())} keys in this dataset")
key_sample = list(data.keys())[0]
first_datapoint = data[key_sample][0]
## Labels ##
print("Labels:")
print(f"Intervals: {first_datapoint["Labels"]["intervals"]}\nFeatures: {first_datapoint["Labels"]["features"]}")
# The features for labels represent the value for ['sentiment','happy','sad','anger','surprise','disgust','fear']
# The value for sentiment is from -3 to 3
# The values for emotions is from 0 to 3.

There are 1185 keys in this dataset
Labels:
Intervals: [ 82.753 100.555]
Features: [1.        0.6666667 0.6666667 0.        0.        0.        0.6666667]


In [20]:
## Audios ##
for i in [0, 1, 2, -3, -2, -1]:
  print(f"Interval {i}: {first_datapoint['COVAREP']['intervals'][i]}")
  print(f"COVAREP: {first_datapoint['COVAREP']['features'][i]}") # there are 74 features for each intervals
  if i == 2:
    print(".....")
    print(".....")
    print(".....")

Interval 0: [82.76 82.77]
COVAREP: [ 1.94500000e+02  0.00000000e+00  7.89933875e-02  4.12973553e-01
  1.13643013e-01  2.47348160e-01  7.90309608e-02 -2.54076809e-01
  1.69600618e+00  3.82096320e-01  1.22102439e-01 -1.02174816e+01
  1.13511670e+00 -9.38661397e-02  1.45152226e-01  1.51863009e-01
 -4.93135415e-02 -4.71199118e-02  1.92586377e-01  6.53149337e-02
  1.80282742e-01 -1.03407525e-01 -4.07525785e-02  5.29282205e-02
  4.68466477e-03 -8.99687335e-02  2.84900088e-02  1.32418990e-01
  5.78956157e-02 -4.16168161e-02 -2.05180459e-02  4.98974957e-02
  6.02267422e-02 -6.16620742e-02  1.94137972e-02  3.92102115e-02
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  2.99928593e-04  1.22163920e-02  9.84863378e-03  2.62905611e-03
 -2.04790253e-02  1.55909350e-02  1.74049914e-01  2.45547414e-01
 -2.83771873e-01  1.26596701e+00 -8.68057668e-01  4.25320752e-02
  1.56559205e+00  3.03968716e+00 -8.18883836e-01 -2.12085557e+00
  1.88038659e+00  1.74354005e+00 -1.41296494e+00 -4.301

In [21]:
## Visual ##
for i in [0, 1, 2, -3, -2, -1]:
  print(f"Interval {i}: {first_datapoint['FACET']['intervals'][i]}")
  print(f"FACET42: {first_datapoint['FACET']['features'][i]}") # there are 35 features for each intervals
  if i == 2:
    print(".....")
    print(".....")
    print(".....")


Interval 0: [82.7667 82.8   ]
FACET42: [-1.21083  -0.461787 -0.511508 -1.27963  -0.700456  1.21737  -1.84845
 -1.29372  -0.887459 -1.06111  -0.285932 -0.27992  -0.649808 -0.806752
 -1.22942  -0.783531 -1.78698  -0.23028  -0.731304 -0.829165 -1.53884
 -1.40423  -1.91299   0.031189 -1.28364  -1.28711  -0.250878 -0.830505
 -1.9368   -0.829042  0.999434  0.978009  2.28579  19.2545   -1.83407 ]
Interval 1: [82.8    82.8333]
FACET42: [-1.49198   -0.311786  -0.389164  -1.6359    -1.69018    1.74129
 -1.6399    -1.82465   -0.608051  -1.35308   -0.459972  -0.471167
 -0.714031  -1.22656   -1.20295   -0.72731   -1.53705   -0.209696
 -0.995856  -0.817197  -1.63576   -1.53025   -1.66863   -0.0265327
 -1.22304   -1.58582   -0.280841  -0.963316  -2.29756   -1.03071
  0.998292   0.984471   1.25417   19.2957    -1.55578  ]
Interval 2: [82.8333 82.8667]
FACET42: [-1.4663   -0.605802 -0.630635 -2.1401   -1.93385   1.80091  -1.67791
 -1.97462  -0.371055 -1.23545  -0.487862 -0.524537 -0.665061 -1.25948
 -1

In [22]:
import torch
import numpy as np
from sklearn.metrics import f1_score

# ---------- Weighted Accuracy ----------
def weighted_accuracy(y_true, y_pred):
    """
    y_true, y_pred: numpy arrays of shape (N, 6) or (N,)
    """
    diff = np.abs(y_true - y_pred)
    weights = 1 - diff / 3
    return np.clip(weights, 0, 1).mean()


# ---------- F1 for emotion presence ----------
def f1_binary_emotion(y_true, y_pred):
    """
    Convert intensities 0-3 to binary (present vs absent)
    """
    y_true_bin = (y_true > 0).astype(int)
    y_pred_bin = (y_pred > 0).astype(int)

    return f1_score(y_true_bin, y_pred_bin, average='weighted')


# ---------- EPISODE / BATCH ACCUMULATION ----------
class MetricTracker:
    def __init__(self):
        self.y_true_all = []
        self.y_pred_all = []

    def update(self, y_true_batch, y_pred_batch):
        """
        y_true_batch: torch tensor (B, 6)
        y_pred_batch: torch tensor (B, 6) logits or raw predictions
        """
        # Convert to CPU numpy safely
        y_true_np = y_true_batch.detach().cpu().numpy()
        y_pred_np = y_pred_batch.detach().cpu().numpy()

        # If model outputs logits/probs, convert to intensity predictions
        # Example: assume model outputs 4-class logits for each emotion (B, 6, 4)
        if y_pred_np.ndim == 3:
            y_pred_np = np.argmax(y_pred_np, axis=2)  # take intensity 0,1,2,3

        self.y_true_all.append(y_true_np)
        self.y_pred_all.append(y_pred_np)

    def compute(self):
        """
        Computes WA and F1 over all batches.
        """
        y_true = np.concatenate(self.y_true_all, axis=0)
        y_pred = np.concatenate(self.y_pred_all, axis=0)

        wa = weighted_accuracy(y_true, y_pred)
        f1 = f1_binary_emotion(y_true, y_pred)

        return wa, f1


Data Splitting, Audio-Visual Model Architecture, Hyperparameter Tuning and Performance Metrics

In [23]:
for key in data:
  data_point = data[key]
  for x in data_point:
    x["Labels"]["features"] = torch.tensor(x["Labels"]["features"][1:])
random.seed(42)
key_list = list(data.keys())
random.shuffle(key_list)
train_keys = key_list[:int(0.7*len(key_list))]
valiation_keys = key_list[int(0.7*len(key_list)):int(0.85*len(key_list))]
test_keys = key_list[int(0.85*len(key_list)):]
train_data = []
val_data = []
test_data = []

# Populate train_data
for vid in train_keys:
    for utterance in data[vid]:
        train_data.append((vid, utterance))

# Populate val_data
for vid in valiation_keys:
    for utterance in data[vid]:
        val_data.append((vid, utterance))

# Populate test_data
for vid in test_keys:
    for utterance in data[vid]:
        test_data.append((vid, utterance))

class MOSEIUttDataset(Dataset):
    """
    Utterance-level CMU-MOSEI dataset.
    items: list of (video_id, utt_dict)
    """
    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        vid, utt = self.items[idx]

        # --- audio ---
        a_feat = utt["COVAREP"]["features"]
        if a_feat.shape[0] == 0:  # If audio feature is empty
            a_feat = np.zeros((1, 74), dtype=np.float32) # Replace with a single zero-padded frame
        a_feat = np.nan_to_num(a_feat, nan=0.0, posinf=0.0, neginf=0.0)
        a_feat = torch.from_numpy(a_feat).float()

        # --- visual ---
        v_feat = utt["FACET"]["features"]
        if v_feat.shape[0] == 0:  # If visual feature is empty
            v_feat = np.zeros((1, 35), dtype=np.float32) # Replace with a single zero-padded frame
        v_feat = np.nan_to_num(v_feat, nan=0.0, posinf=0.0, neginf=0.0)
        v_feat = torch.from_numpy(v_feat).float()

        # --- label ---
        y = utt["Labels"]["features"]
        y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
        y = torch.from_numpy(y)

        sample = {
            "video_id": vid,
            "audio": a_feat,     # (T_a, 74)
            "visual": v_feat,    # (T_v, 35)
            "label": y           # (7,)
        }

        return sample
def mosei_collate_fn(batch):
    """
    batch: list of dicts from MOSEIUttDataset.__getitem__
    Returns padded tensors and masks.
    """
    # lengths
    a_lengths = [b["audio"].shape[0] for b in batch]
    v_lengths = [b["visual"].shape[0] for b in batch]

    max_a = max(a_lengths)
    max_v = max(v_lengths)

    B = len(batch)
    a_dim = batch[0]["audio"].shape[1]
    v_dim = batch[0]["visual"].shape[1]

    # padded tensors
    audio_batch = torch.zeros(B, max_a, a_dim, dtype=torch.float32)
    visual_batch = torch.zeros(B, max_v, v_dim, dtype=torch.float32)

    # masks: 1 for real frame, 0 for padding
    audio_mask = torch.zeros(B, max_a, dtype=torch.bool)
    visual_mask = torch.zeros(B, max_v, dtype=torch.bool)

    labels = []
    video_ids = []

    for i, b in enumerate(batch):
        Ta = b["audio"].shape[0]
        Tv = b["visual"].shape[0]

        audio_batch[i, :Ta] = b["audio"]
        visual_batch[i, :Tv] = b["visual"]

        audio_mask[i, :Ta] = True
        visual_mask[i, :Tv] = True

        labels.append(b["label"])
        video_ids.append(b["video_id"])

    labels = torch.stack(labels, dim=0)  # (B, 7)

    return {
        "video_id": video_ids,
        "audio": audio_batch,        # (B, max_Ta, 74)
        "audio_mask": audio_mask,    # (B, max_Ta)
        "visual": visual_batch,      # (B, max_Tv, 35)
        "visual_mask": visual_mask,  # (B, max_Tv)
        "label": labels              # (B, 7)
    }
batch_size = 5

train_dataset = MOSEIUttDataset(train_data)
val_dataset   = MOSEIUttDataset(val_data)
test_dataset  = MOSEIUttDataset(test_data)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=mosei_collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=mosei_collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=mosei_collate_fn
)
# ---------- Positional Encoding ----------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)      # (T, D)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)                    # (1, T, D)
        self.register_buffer("pe", pe)
        self.dropout = nn.Dropout(p=0.1)
    def forward(self, x):
        """
        x: (B, T, D)
        """
        T = x.size(1)
        return self.dropout(x + self.pe[:, :T, :])


# ---------- Masked temporal mean pooling ----------
def masked_mean(x, mask):
    """
    x:    (B, T, D)
    mask: (B, T) with 1 for valid, 0 for pad
    """
    mask = mask.unsqueeze(-1)          # (B, T, 1)
    x = x * mask                       # zero out pads
    lengths = mask.sum(dim=1).clamp(min=1)   # (B, 1)
    return x.sum(dim=1) / lengths      # (B, D)

def masked_max(x, mask):
    """
    x:    (B, T, D)
    mask: (B, T) with 1 for valid, 0 for pad
    """
    # Set padded elements to a very small negative number so they are ignored by max
    x_masked = x.masked_fill(~mask.unsqueeze(-1), -torch.inf) # (B, T, D)
    return x_masked.max(dim=1)[0] # (B, D), [0] to get the values, not indices


# ---------- Main multimodal model ----------
class AVTransformer(nn.Module):
    def __init__(
        self,
        audio_in_dim=74,
        visual_in_dim=35,
        d_model=128,
        n_heads=4,
        num_layers=2,
        ff_dim=256,
        dropout=0.1,
        num_labels=7,
    ):
        super().__init__()

        # Projections
        self.audio_proj = nn.Sequential(
            nn.Linear(audio_in_dim, d_model),
            nn.Dropout(dropout)
        )
        self.visual_proj = nn.Sequential(
            nn.Linear(visual_in_dim, d_model),
            nn.Dropout(dropout)
        )

        self.pos_enc = PositionalEncoding(d_model)

        # Unimodal encoders
        encoder_layer_audio = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,  # important: inputs (B, T, D)
        )
        encoder_layer_visual = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,  # important: inputs (B, T, D)
        )
        self.audio_encoder = nn.TransformerEncoder(encoder_layer_audio, num_layers=num_layers)
        self.visual_encoder = nn.TransformerEncoder(encoder_layer_visual, num_layers=num_layers)

        # Cross-attention modules
        self.cross_attn_a2v = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=n_heads, dropout=dropout, batch_first=True
        )
        self.cross_attn_v2a = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=n_heads, dropout=dropout, batch_first=True
        )

        self.cross_ln_audio = nn.LayerNorm(d_model)
        self.cross_ln_visual = nn.LayerNorm(d_model)
        self.cross_dropout = nn.Dropout(dropout)
        # "Transformation encoder" (simple 2-layer MLP here)
        self.trans_encoder = nn.Sequential(
            nn.Linear(2 * d_model, ff_dim),
            nn.LayerNorm(ff_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, d_model),
            nn.LayerNorm(d_model)
        )

        # Final output layer
        self.fc_out = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d_model, num_labels)
        )

    def forward(self, audio, audio_mask, visual, visual_mask):
        """
        audio:       (B, Ta, 74)
        audio_mask:  (B, Ta)  1=valid, 0=pad
        visual:      (B, Tv, 35)
        visual_mask: (B, Tv)
        """

        # --- Audio branch ---
        a = self.audio_proj(audio)          # (B, Ta, D)
        a = self.pos_enc(a)
        # key_padding_mask expects True for PAD, so invert:
        a_kpm = (audio_mask == 0)           # (B, Ta)
        a = self.audio_encoder(a, src_key_padding_mask=a_kpm)  # (B, Ta, D)

        # --- Visual branch ---
        v = self.visual_proj(visual)        # (B, Tv, D)
        v = self.pos_enc(v)
        v_kpm = (visual_mask == 0)          # (B, Tv)
        v = self.visual_encoder(v, src_key_padding_mask=v_kpm) # (B, Tv, D)

        # --- Cross-attention: audio attends to visual ---
        a_cross, _ = self.cross_attn_a2v(
            query=a,
            key=v,
            value=v,
            key_padding_mask=v_kpm,   # pads for key/value
        )
        a = self.cross_ln_audio(a + self.cross_dropout(a_cross))

        # --- Cross-attention: visual attends to audio ---
        v_cross, _ = self.cross_attn_v2a(
            query=v,
            key=a,
            value=a,
            key_padding_mask=a_kpm,
        )
        v = self.cross_ln_visual(v + self.cross_dropout(v_cross))

        # --- Temporal pooling (masked mean) ---
        a_pooled = masked_mean(a, audio_mask)   # (B, D)
        v_pooled = masked_mean(v, visual_mask)  # (B, D)

        # --- Fusion + transformation encoder ---
        fused = torch.cat([a_pooled, v_pooled], dim=-1)  # (B, 2D)
        h = self.trans_encoder(fused)                    # (B, D)

        # --- Output ---
        out = self.fc_out(h)       # (B, 7)
        return out
def train_epoch(model, dataloader, optimizer, criterion, device, scaler, metric):
    model.train() # Set the model to training mode
    total_loss = 0
    for batch in dataloader:
        audio = batch['audio'].to(device)
        audio_mask = batch['audio_mask'].to(device)
        visual = batch['visual'].to(device)
        visual_mask = batch['visual_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad() # Zero the gradients

        with torch.amp.autocast('cuda'): # Updated: Use torch.amp.autocast('cuda')
            outputs = model(audio, audio_mask, visual, visual_mask)
            loss = criterion(outputs, labels)
            metric.update(labels, outputs)

        # Scale the loss and call backward()
        scaler.scale(loss).backward()

        # Unscale gradients and step the optimizer
        scaler.step(optimizer)

        # Update the scaler for the next iteration
        scaler.update()


        total_loss += loss.item()

    return total_loss / len(dataloader)

def evaluate_model(model, dataloader, criterion, device, metric):
    model.eval() # Set the model to evaluation mode
    total_loss = 0
    with torch.no_grad(): # Disable gradient calculation
        for batch in dataloader:
            audio = batch['audio'].to(device)
            audio_mask = batch['audio_mask'].to(device)
            visual = batch['visual'].to(device)
            visual_mask = batch['visual_mask'].to(device)
            labels = batch['label'].to(device)

            with torch.amp.autocast('cuda'): # Updated: Use torch.amp.autocast('cuda')
                outputs = model(audio, audio_mask, visual, visual_mask)
                loss = criterion(outputs, labels)
                metric.update(labels, outputs)
            total_loss += loss.item()
        wa, f1 = metric.compute()
    return wa, f1, total_loss / len(dataloader)

In [24]:
param_grid = {
    'learning_rate': [1e-4],  # a bit coarse but covers "normal" and "small"
    'num_layers': [1, 2],           # shallow vs slightly deeper
    'd_model': [32, 64],           # smaller vs medium - Adjusted for memory
    'n_heads': [2, 4],              # compatible with both 32 and 64
}
hyperparameter_combinations = list(itertools.product(*param_grid.values()))
num_epochs = 4  # Initialize num_epochs
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') # Initialize device

best_val_loss = float('inf') # Initialize best_val_loss
best_params = None # Initialize best_params
best_model_state_dict = None # Initialize best_model_state_dict
best_acc = 0

In [27]:
from numpy.matrixlib import test
import math
import torch.optim as optim
import torch.cuda.amp as amp # Import the amp module
from sklearn.metrics import f1_score # Added for F1 score calculation

# Fixed dimensions for the model
audio_in_dim = 74
visual_in_dim = 35
num_labels = 6
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') # Initialize device

param_grid = {
    'learning_rate': [1e-4],  # a bit coarse but covers "normal" and "small"
    'num_layers': [1, 2],           # shallow vs slightly deeper
    'd_model': [32, 64],           # smaller vs medium - Adjusted for memory
    'n_heads': [2, 4],              # compatible with both 32 and 64
}
hyperparameter_combinations = list(itertools.product(*param_grid.values()))
num_epochs = 4  # Initialize num_epochs

best_val_loss = float('inf') # Initialize best_val_loss (still useful for general tracking)
best_f1_score_val = 0.0 # Initialize best_f1_score_val to prioritize F1 for model selection
best_wa = 0.0 # Initialize best_acc
best_params = None # Initialize best_params
best_model_state_dict = None # Initialize best_model_state_dict
best_av_model = None # Initialize best_model
# Instantiate best_model with increased complexity hyperparameters
for i, combo in enumerate(hyperparameter_combinations):
    current_params = dict(zip(param_grid.keys(), combo))

    print(f"--- Combination {i+1}/{len(hyperparameter_combinations)} ---")
    print("Current hyperparameters:")
    for param, value in current_params.items():
        print(f"  {param}: {value}")

    d_model = current_params['d_model']
    ff_dim = 2 * d_model

    scaler = torch.amp.GradScaler('cuda') # Updated: Use torch.amp.GradScaler('cuda')

    # Instantiate the model with current hyperparameters
    model = AVTransformer(
        audio_in_dim=audio_in_dim,
        visual_in_dim=visual_in_dim,
        d_model=d_model,
        n_heads=current_params['n_heads'],
        num_layers=current_params['num_layers'],
        ff_dim=ff_dim,
        num_labels=num_labels
    ).to(device) # Move model to device

    # Define loss function and optimizer
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=current_params['learning_rate']) # Initialize optimizer

    print(f"Model instantiated and moved to {device}.")
    print(f"Starting training for {num_epochs} epochs...")

    # Training loop for current combination
    for epoch in range(num_epochs):
        train_metric = MetricTracker()
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device, scaler, train_metric)
        train_wa, train_f1 = train_metric.compute()
        val_metric = MetricTracker()
        val_loss = evaluate_model(model, val_loader, criterion, device, val_metric)
        val_wa, val_f1, val_loss = evaluate_model(model, val_loader, criterion, device, val_metric)
        print(f"    Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Train Weight Average: {train_wa:.4f}, Train F1: {train_f1:.4f}")
        print(f"    Epoch {epoch+1}/{num_epochs}, Val Loss: {val_loss:.4f}, Val Weight Average: {val_wa:.4f}, Val F1: {val_f1:.4f}")

        # Track best model based on F1 score (using validation F1)
        if val_f1 > best_f1_score_val:
            best_f1_score_val = val_f1
            best_wa = val_wa
            best_val_loss = val_loss # Also store current validation loss
            best_params = current_params.copy() # Store a copy of params
            best_model_state_dict = model.state_dict() # Store model state
            best_av_model = model # Store model instance
            print(f"      New best Validation F1 score: {best_f1_score_val:.4f}")
    print("\n") # Add a newline for better readability between combinations

print("Hyperparameter search complete.")
print(f"Best F1 score found on validation set: {best_f1_score_val:.4f}")
print(f"Validation loss for that F1: {best_val_loss:.4f}")
print(f"Best hyperparameters: {best_params}")

test_metric = MetricTracker()
test_wa, test_f1, test_loss = evaluate_model(best_av_model, test_loader, criterion, device, test_metric)
print(f"Test Weighted Average: {test_wa}, Test F1: {test_f1}, Test loss: {test_loss}")

--- Combination 1/8 ---
Current hyperparameters:
  learning_rate: 0.0001
  num_layers: 1
  d_model: 32
  n_heads: 2
Model instantiated and moved to cuda.
Starting training for 4 epochs...
    Epoch 1/4, Train Loss: 0.1462, Train Weight Average: 0.9113, Train F1: 0.4404
    Epoch 1/4, Val Loss: 0.1243, Val Weight Average: 0.9285, Val F1: 0.4387
      New best Validation F1 score: 0.4387
    Epoch 2/4, Train Loss: 0.1158, Train Weight Average: 0.9251, Train F1: 0.4493
    Epoch 2/4, Val Loss: 0.1226, Val Weight Average: 0.9201, Val F1: 0.4396
      New best Validation F1 score: 0.4396
    Epoch 3/4, Train Loss: 0.1089, Train Weight Average: 0.9284, Train F1: 0.4529
    Epoch 3/4, Val Loss: 0.1226, Val Weight Average: 0.9271, Val F1: 0.4397
      New best Validation F1 score: 0.4397
    Epoch 4/4, Train Loss: 0.1061, Train Weight Average: 0.9298, Train F1: 0.4529
    Epoch 4/4, Val Loss: 0.1206, Val Weight Average: 0.9281, Val F1: 0.4398
      New best Validation F1 score: 0.4398


--- Co

Data Splitting, Audio-Visual-Text Model Architecture, Hyperparameter Tuning and Performance Metrics

In [28]:
with open("/content/processed_mosei.pkl", "rb") as f:
  data = pickle.load(f)
for key in data:
  data_point = data[key]
  for x in data_point:
    x["Labels"]["features"] = torch.tensor(x["Labels"]["features"][1:])
random.seed(42)
key_list = list(data.keys())
random.shuffle(key_list)
train_keys = key_list[:int(0.7*len(key_list))]
valiation_keys = key_list[int(0.7*len(key_list)):int(0.85*len(key_list))]
test_keys = key_list[int(0.85*len(key_list)):]
train_data = []
val_data = []
test_data = []

# Populate train_data
for vid in train_keys:
    for utterance in data[vid]:
        train_data.append((vid, utterance))

# Populate val_data
for vid in valiation_keys:
    for utterance in data[vid]:
        val_data.append((vid, utterance))

# Populate test_data
for vid in test_keys:
    for utterance in data[vid]:
        test_data.append((vid, utterance))

class MOSEI_AVT_Dataset(Dataset):
    """
    Utterance-level CMU-MOSEI dataset.
    items: list of (video_id, utt_dict)
    """
    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        vid, utt = self.items[idx]

        # --- audio ---
        a_feat = utt["COVAREP"]["features"]
        if a_feat.shape[0] == 0:  # If audio feature is empty
            a_feat = np.zeros((1, 74), dtype=np.float32) # Replace with a single zero-padded frame
        a_feat = np.nan_to_num(a_feat, nan=0.0, posinf=0.0, neginf=0.0)
        a_feat = torch.from_numpy(a_feat).float()

        # --- visual ---
        v_feat = utt["FACET"]["features"]
        if v_feat.shape[0] == 0:  # If visual feature is empty
            v_feat = np.zeros((1, 35), dtype=np.float32) # Replace with a single zero-padded frame
        v_feat = np.nan_to_num(v_feat, nan=0.0, posinf=0.0, neginf=0.0)
        v_feat = torch.from_numpy(v_feat).float()

        # --- text ---
        t_feat = utt["WORDVEC"]["features"]
        if t_feat.shape[0] == 0:  # If visual feature is empty
            t_feat = np.zeros((1, 300), dtype=np.float32) # Replace with a single zero-padded frame
        t_feat = np.nan_to_num(t_feat, nan=0.0, posinf=0.0, neginf=0.0)
        t_feat = torch.from_numpy(t_feat).float()

        # --- label ---
        y = utt["Labels"]["features"]
        y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
        y = torch.from_numpy(y)

        sample = {
            "video_id": vid,
            "audio": a_feat,     # (T_a, 74)
            "visual": v_feat,    # (T_v, 35)
            "text": t_feat,
            "label": y           # (7,)
        }

        return sample
def mosei_collate_fn(batch):
    """
    batch: list of dicts from MOSEIUttDataset.__getitem__
    Returns padded tensors and masks.
    """
    # lengths
    a_lengths = [b["audio"].shape[0] for b in batch]
    v_lengths = [b["visual"].shape[0] for b in batch]
    t_lengths = [b["text"].shape[0] for b in batch]

    max_a = max(a_lengths)
    max_v = max(v_lengths)
    max_t = max(t_lengths)

    B = len(batch)
    a_dim = batch[0]["audio"].shape[1]
    v_dim = batch[0]["visual"].shape[1]
    t_dim = batch[0]["text"].shape[1]

    # padded tensors
    audio_batch = torch.zeros(B, max_a, a_dim, dtype=torch.float32)
    visual_batch = torch.zeros(B, max_v, v_dim, dtype=torch.float32)
    text_batch = torch.zeros(B, max_t, t_dim, dtype=torch.float32)

    # masks: 1 for real frame, 0 for padding
    audio_mask = torch.zeros(B, max_a, dtype=torch.bool)
    visual_mask = torch.zeros(B, max_v, dtype=torch.bool)
    text_mask = torch.zeros(B, max_t, dtype=torch.bool)

    labels = []
    video_ids = []

    for i, b in enumerate(batch):
        Ta = b["audio"].shape[0]
        Tv = b["visual"].shape[0]
        Tt = b["text"].shape[0]

        audio_batch[i, :Ta] = b["audio"]
        visual_batch[i, :Tv] = b["visual"]
        text_batch[i, :Tt] = b["text"]

        audio_mask[i, :Ta] = True
        visual_mask[i, :Tv] = True
        text_mask[i, :Tt] = True

        labels.append(b["label"])
        video_ids.append(b["video_id"])

    labels = torch.stack(labels, dim=0)  # (B, 7)

    return {
        "video_id": video_ids,
        "audio": audio_batch,        # (B, max_Ta, 74)
        "audio_mask": audio_mask,    # (B, max_Ta)
        "visual": visual_batch,      # (B, max_Tv, 35)
        "visual_mask": visual_mask,  # (B, max_Tv)
        "text": text_batch,          # (B, max_Tv, 300)
        "text_mask": text_mask,
        "label": labels              # (B, 7)
    }
batch_size = 5

train_dataset = MOSEI_AVT_Dataset(train_data)
val_dataset   = MOSEI_AVT_Dataset(val_data)
test_dataset  = MOSEI_AVT_Dataset(test_data)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=mosei_collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=mosei_collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=mosei_collate_fn
)
# ---------- Positional Encoding ----------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)      # (T, D)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)                    # (1, T, D)
        self.register_buffer("pe", pe)
        self.dropout = nn.Dropout(p=0.1)
    def forward(self, x):
        """
        x: (B, T, D)
        """
        T = x.size(1)
        return self.dropout(x + self.pe[:, :T, :])


# ---------- Masked temporal mean pooling ----------
def masked_mean(x, mask):
    """
    x:    (B, T, D)
    mask: (B, T) with 1 for valid, 0 for pad
    """
    mask = mask.unsqueeze(-1)          # (B, T, 1)
    x = x * mask                       # zero out pads
    lengths = mask.sum(dim=1).clamp(min=1)   # (B, 1)
    return x.sum(dim=1) / lengths      # (B, D)

def masked_max(x, mask):
    """
    x:    (B, T, D)
    mask: (B, T) with 1 for valid, 0 for pad
    """
    # Set padded elements to a very small negative number so they are ignored by max
    x_masked = x.masked_fill(~mask.unsqueeze(-1), -torch.inf) # (B, T, D)
    return x_masked.max(dim=1)[0] # (B, D), [0] to get the values, not indices


In [29]:
print(len(data["-3g5yACwYnA"][0]["WORDVEC"]["features"][0]))

300


In [30]:
class AVTTransformer(nn.Module):
    def __init__(
        self,
        audio_in_dim=74,
        visual_in_dim=35,
        text_in_dim=300,
        d_model=128,
        n_heads=4,
        num_layers=2,
        ff_dim=256,
        dropout=0.1,
        num_labels=6,
    ):
        super().__init__()

        # Projections
        self.audio_proj = nn.Sequential(
            nn.Linear(audio_in_dim, d_model),
            nn.Dropout(dropout)
        )
        self.visual_proj = nn.Sequential(
            nn.Linear(visual_in_dim, d_model),
            nn.Dropout(dropout)
        )
        self.text_proj = nn.Sequential(
            nn.Linear(text_in_dim, d_model),
            nn.Dropout(dropout)
        )

        self.pos_enc = PositionalEncoding(d_model)

        # Unimodal encoders
        encoder_layer_audio = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,  # important: inputs (B, T, D)
        )
        encoder_layer_visual = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,  # important: inputs (B, T, D)
        )
        encoder_layer_text = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,  # important: inputs (B, T, D)
        )
        self.audio_encoder = nn.TransformerEncoder(encoder_layer_audio, num_layers=num_layers)
        self.visual_encoder = nn.TransformerEncoder(encoder_layer_visual, num_layers=num_layers)
        self.text_encoder = nn.TransformerEncoder(encoder_layer_text, num_layers=num_layers)

        # Cross-attention modules
        self.cross_attn_a2v = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=n_heads, dropout=dropout, batch_first=True
        )
        self.cross_attn_a2t = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=n_heads, dropout=dropout, batch_first=True
        )
        self.cross_attn_v2t = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=n_heads, dropout=dropout, batch_first=True
        )
        self.cross_attn_v2a = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=n_heads, dropout=dropout, batch_first=True
        )
        self.cross_attn_t2a = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=n_heads, dropout=dropout, batch_first=True
        )
        self.cross_attn_t2v = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=n_heads, dropout=dropout, batch_first=True
        )

        self.cross_ln_audio = nn.LayerNorm(d_model)
        self.cross_ln_visual = nn.LayerNorm(d_model)
        self.cross_ln_text = nn.LayerNorm(d_model)
        self.cross_dropout = nn.Dropout(dropout)
        # "Transformation encoder" (simple 2-layer MLP here)
        self.trans_encoder = nn.Sequential(
            nn.Linear(3 * d_model, ff_dim),
            nn.LayerNorm(ff_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, d_model),
            nn.LayerNorm(d_model)
        )

        # Final output layer
        self.fc_out = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d_model, num_labels)
        )

    def forward(self, audio, audio_mask, visual, visual_mask, text, text_mask):
        """
        audio:       (B, Ta, 74)
        audio_mask:  (B, Ta)  1=valid, 0=pad
        visual:      (B, Tv, 35)
        visual_mask: (B, Tv)
        """

        # --- Audio branch ---
        a = self.audio_proj(audio)          # (B, Ta, D)
        a = self.pos_enc(a)
        # key_padding_mask expects True for PAD, so invert:
        a_kpm = (audio_mask == 0)           # (B, Ta)
        a = self.audio_encoder(a, src_key_padding_mask=a_kpm)  # (B, Ta, D)

        # --- Visual branch ---
        v = self.visual_proj(visual)        # (B, Tv, D)
        v = self.pos_enc(v)
        v_kpm = (visual_mask == 0)          # (B, Tv)
        v = self.visual_encoder(v, src_key_padding_mask=v_kpm) # (B, Tv, D)

        # --- Text branch ---
        t = self.text_proj(text)        # (B, Tv, D)
        t = self.pos_enc(t)
        t_kpm = (text_mask == 0)          # (B, Tv)
        t = self.text_encoder(t, src_key_padding_mask=t_kpm) # (B, Tv, D)

        # --- Cross-attention: audio attends to visual ---
        a_v, _ = self.cross_attn_a2v(
            query=a,
            key=v,
            value=v,
            key_padding_mask=v_kpm,   # pads for key/value
        )
        a_t, _ = self.cross_attn_a2t(
            query=a,
            key=t,
            value=t,
            key_padding_mask=t_kpm,
        )
        a = self.cross_ln_audio(a + self.cross_dropout(a_v) + self.cross_dropout(a_t))

        # --- Cross-attention: visual attends to audio ---
        v_a, _ = self.cross_attn_v2a(
            query=v,
            key=a,
            value=a,
            key_padding_mask=a_kpm,
        )
        v_t, _ = self.cross_attn_v2t(
            query=v,
            key=t,
            value=t,
            key_padding_mask=t_kpm,
        )
        v = self.cross_ln_visual(v + self.cross_dropout(v_a) + self.cross_dropout(v_t))

        # --- Cross-attention: text attends to audio ---
        t_a, _ = self.cross_attn_t2a(
            query=t,
            key=a,
            value=a,
            key_padding_mask=a_kpm,
        )
        t_v, _ = self.cross_attn_t2v(
            query=t,
            key=v,
            value=v,
            key_padding_mask=v_kpm,
        )
        t = self.cross_ln_text(t + self.cross_dropout(t_a) + self.cross_dropout(t_v))

        # --- Temporal pooling (masked mean) ---
        a_pooled = masked_mean(a, audio_mask)   # (B, D)
        v_pooled = masked_mean(v, visual_mask)  # (B, D)
        t_pooled = masked_mean(t, text_mask)  # (B, D)
        # --- Fusion + transformation encoder ---
        fused = torch.cat([a_pooled, v_pooled, t_pooled], dim=-1)  # (B, 2D)
        h = self.trans_encoder(fused)                    # (B, D)

        # --- Output ---
        out = self.fc_out(h)       # (B, 7)
        return out


In [31]:
def train_epoch(model, dataloader, optimizer, criterion, device, scaler, metric):
    model.train() # Set the model to training mode
    total_loss = 0
    for batch in dataloader:
        audio = batch['audio'].to(device)
        audio_mask = batch['audio_mask'].to(device)
        visual = batch['visual'].to(device)
        visual_mask = batch['visual_mask'].to(device)
        text = batch['text'].to(device)
        text_mask = batch['text_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad() # Zero the gradients

        with torch.amp.autocast('cuda'): # Updated: Use torch.amp.autocast('cuda')
            outputs = model(audio, audio_mask, visual, visual_mask, text, text_mask)
            loss = criterion(outputs, labels)
            metric.update(outputs, labels)

        # Scale the loss and call backward()
        scaler.scale(loss).backward()

        # Unscale gradients and step the optimizer
        scaler.step(optimizer)

        # Update the scaler for the next iteration
        scaler.update()

        total_loss += loss.item()

    return total_loss / len(dataloader)

def evaluate_model(model, dataloader, criterion, device, metric):
    model.eval() # Set the model to evaluation mode
    total_loss = 0
    with torch.no_grad(): # Disable gradient calculation
        for batch in dataloader:
            audio = batch['audio'].to(device)
            audio_mask = batch['audio_mask'].to(device)
            visual = batch['visual'].to(device)
            visual_mask = batch['visual_mask'].to(device)
            text = batch['text'].to(device)
            text_mask = batch['text_mask'].to(device)
            labels = batch['label'].to(device)

            with torch.amp.autocast('cuda'): # Updated: Use torch.amp.autocast('cuda')
                outputs = model(audio, audio_mask, visual, visual_mask, text, text_mask)
                loss = criterion(outputs, labels)
                metric.update(outputs, labels)
            total_loss += loss.item()
        wa, f1 = metric.compute()
    return wa, f1, total_loss / len(dataloader)

In [36]:
import math
import torch.optim as optim
import torch.cuda.amp as amp # Import the amp module
from sklearn.metrics import f1_score # Added for F1 score calculation

param_grid = {
    'learning_rate': [1e-4],  # a bit coarse but covers "normal" and "small"
    'num_layers': [1, 2],           # shallow vs slightly deeper
    'd_model': [32, 64],           # smaller vs medium - Adjusted for memory
    'n_heads': [2],              # compatible with both 32 and 64
}
hyperparameter_combinations = list(itertools.product(*param_grid.values()))
num_epochs = 4  # Initialize num_epochs
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') # Initialize device

best_val_loss_avt = float('inf') # Initialize best_val_loss (still useful for general tracking)
best_f1_score_val_avt = 0.0 # Initialize best_f1_score_val to prioritize F1 for model selection
best_wa_avt = 0.0 # Initialize best_acc
best_params_avt = None # Initialize best_params
best_model_state_dict_avt = None # Initialize best_model_state_dict
# Fixed dimensions for the model
audio_in_dim = 74
visual_in_dim = 35
text_in_dim = 300
num_labels = 6
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') # Initialize device

# Instantiate best_model with increased complexity hyperparameters
for i, combo in enumerate(hyperparameter_combinations):
    current_params = dict(zip(param_grid.keys(), combo))

    print(f"--- Combination {i+1}/{len(hyperparameter_combinations)} ---")
    print("Current hyperparameters:")
    for param, value in current_params.items():
        print(f"  {param}: {value}")

    d_model = current_params['d_model']
    ff_dim = 2 * d_model

    scaler = torch.amp.GradScaler('cuda') # Updated: Use torch.amp.GradScaler('cuda')

    # Instantiate the model with current hyperparameters
    model = AVTTransformer(
        audio_in_dim=audio_in_dim,
        visual_in_dim=visual_in_dim,
        text_in_dim=text_in_dim,
        d_model=d_model,
        n_heads=current_params['n_heads'],
        num_layers=current_params['num_layers'],
        ff_dim=ff_dim,
        num_labels=num_labels
    ).to(device) # Move model to device

    # Define loss function and optimizer
    criterion = nn.MSELoss() # Define nn.MSELoss
    optimizer = optim.Adam(model.parameters(), lr=current_params['learning_rate']) # Initialize optimizer

    print(f"Model instantiated and moved to {device}.")
    print(f"Starting training for {num_epochs} epochs...")

    # Training loop for current combination
    for epoch in range(num_epochs):
        train_metric = MetricTracker()
        val_metric = MetricTracker()
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device, scaler, train_metric)
        train_wa, train_f1 = train_metric.compute()
        val_wa, val_f1, val_loss = evaluate_model(model, val_loader, criterion, device, val_metric)
        print(f"    Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Train Weight Average: {train_wa:.4f}, Train F1: {train_f1:.4f}")
        print(f"    Epoch {epoch+1}/{num_epochs}, Val Loss: {val_loss:.4f}, Val Weight Average: {val_wa:.4f}, Val F1: {val_f1:.4f}")

        # Track best model based on F1 score (using validation F1)
        if val_f1 > best_f1_score_val_avt:
            best_f1_score_val_avt = val_f1
            best_wa_avt = val_wa # Also store current validation accuracy
            best_val_loss_avt = val_loss # Also store current validation loss
            best_params_avt = current_params.copy() # Store a copy of params
            best_model_state_dict_avt = model.state_dict() # Store model state
            print(f"      New best Validation F1 score: {best_f1_score_val_avt:.4f}")
    print("\n") # Add a newline for better readability between combinations

print("Hyperparameter search complete.")
print(f"Best F1 score found on validation set: {best_f1_score_val_avt:.4f}")
print(f"Validation loss for that F1: {best_val_loss_avt:.4f}")
print(f"Best hyperparameters: {best_params_avt}")

# Optional: Instantiate the best model
if best_model_state_dict_avt is not None and best_params_avt is not None:
    d_model = best_params_avt['d_model']
    ff_dim = 2 * d_model

    # Note: This best_model is only for storing the state dict, it will be used later for final test evaluation.
    best_model_avt = AVTTransformer( # Corrected to AVTTransformer
        audio_in_dim=audio_in_dim,
        visual_in_dim=visual_in_dim,
        text_in_dim=text_in_dim,
        d_model=d_model,
        n_heads=best_params_avt['n_heads'],
        num_layers=best_params_avt['num_layers'],
        ff_dim=ff_dim,
        num_labels=num_labels
    ).to(device)
    best_model_avt.load_state_dict(best_model_state_dict_avt)
    print("Best AVT model state dict loaded successfully.")
else:
    print("No best AVT model found (this should not happen if combinations were explored).")

--- Combination 1/4 ---
Current hyperparameters:
  learning_rate: 0.0001
  num_layers: 1
  d_model: 32
  n_heads: 2
Model instantiated and moved to cuda.
Starting training for 4 epochs...
    Epoch 1/4, Train Loss: 0.1511, Train Weight Average: 0.9104, Train F1: 0.3841
    Epoch 1/4, Val Loss: 0.1215, Val Weight Average: 0.9287, Val F1: 0.3916
      New best Validation F1 score: 0.3916
    Epoch 2/4, Train Loss: 0.1162, Train Weight Average: 0.9245, Train F1: 0.3896
    Epoch 2/4, Val Loss: 0.1197, Val Weight Average: 0.9259, Val F1: 0.3913
    Epoch 3/4, Train Loss: 0.1077, Train Weight Average: 0.9283, Train F1: 0.3933
    Epoch 3/4, Val Loss: 0.1155, Val Weight Average: 0.9315, Val F1: 0.3945
      New best Validation F1 score: 0.3945
    Epoch 4/4, Train Loss: 0.1023, Train Weight Average: 0.9308, Train F1: 0.3970
    Epoch 4/4, Val Loss: 0.1183, Val Weight Average: 0.9304, Val F1: 0.4001
      New best Validation F1 score: 0.4001


--- Combination 2/4 ---
Current hyperparameters:


Evaluating the Best Models on the Test Set

In [37]:
test_metric = MetricTracker()
criterion = nn.MSELoss()
test_wa, test_f1, test_loss = evaluate_model(best_model_avt, test_loader, criterion, device, test_metric)

In [38]:
print(f"Test Weight Average: {test_wa}, Test F1: {test_f1}, Test loss: {test_loss}")

Test Weight Average: 0.9326238036155701, Test F1: 0.3960879469156383, Test loss: 0.1077646554565464
